# Tutorial 9: Why Local Methods Fail

**Prerequisites:** T3 (Gradient Descent), T5 (Newton/BFGS)  
**Up Next:** T10 — Genetic Algorithms

---

## Concept

In Tutorials 3 and 5 we learned gradient descent and BFGS — powerful **local** methods that follow the gradient downhill to a minimum. But which minimum? Real engineering landscapes like four-bar path synthesis have **many** local minima. A local method converges to whichever basin its starting point falls in, and different starting points yield different (often poor) solutions.

This tutorial demonstrates the problem and motivates **global** optimization methods that search the entire landscape.

## Key Takeaway

> **Local methods find *a* minimum, not *the* minimum. When the landscape has many local minima, we need global methods that explore more broadly.**

## Math Core

A point $\mathbf{x}^*$ is a **local minimum** if $f(\mathbf{x}^*) \le f(\mathbf{x})$ for all $\mathbf{x}$ in some neighbourhood. It is a **global minimum** if this holds for *all* feasible $\mathbf{x}$.

Gradient-based methods guarantee convergence to a local minimum (under mild conditions), but they have **no mechanism** to jump between basins:

$$\mathbf{x}_{k+1} = \mathbf{x}_k - \alpha_k \, H_k^{-1} \nabla f(\mathbf{x}_k)$$

The trajectory is entirely determined by the starting point $\mathbf{x}_0$.

## Code

In [ ]:
from dms.mechanisms.fourbar import FourBar
from dms.curves import CompareCurves
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize
%matplotlib inline

# Fixed link lengths
L0, L1 = 1.0, 1.0
# Coupler point offset
PX, PY = 0.5, 0.3
# Target path
TARGETS = np.array([
    [1.0, 1.5], [0.75, 1.5], [0.5, 1.5],
    [0.25, 1.5], [0.0, 1.5], [-0.25, 1.5]
])
THETAS = np.linspace(np.deg2rad(60), np.deg2rad(120), len(TARGETS))

In [ ]:
def fourbar_fk(theta1, l0, l1, l2, l3):
    """Solve four-bar loop closure for passive angles (theta2, theta3)."""
    ax, ay = l1 * np.cos(theta1), l1 * np.sin(theta1)
    dx, dy = ax - l0, ay
    d = np.sqrt(dx**2 + dy**2)
    cos_alpha = (l3**2 + d**2 - l2**2) / (2 * l3 * d)
    if np.abs(cos_alpha) > 1:
        return None
    alpha = np.arccos(np.clip(cos_alpha, -1, 1))
    beta = np.arctan2(dy, dx)
    theta3 = beta + alpha
    bx = l0 + l3 * np.cos(theta3)
    by = l3 * np.sin(theta3)
    theta2 = np.arctan2(by - ay, bx - ax)
    return theta2, theta3


def coupler_point(theta1, l2, l3):
    """Compute coupler point position for given crank angle and link lengths."""
    sol = fourbar_fk(theta1, L0, L1, l2, l3)
    if sol is None:
        return None
    theta2, _ = sol
    ax, ay = L1 * np.cos(theta1), L1 * np.sin(theta1)
    ux, uy = np.cos(theta2), np.sin(theta2)
    return np.array([ax + PX * ux - PY * uy,
                     ay + PX * uy + PY * ux])


def objective(x):
    """Objective f(x): how far is the coupler path from the targets?"""
    l2, l3 = x
    path = []
    for theta1 in THETAS:
        pt = coupler_point(theta1, l2, l3)
        if pt is None:
            return 1e3
        path.append(pt)
    path = np.array(path)
    return CompareCurves(path[:, 0], path[:, 1],
                         TARGETS[:, 0], TARGETS[:, 1])

### The four-bar landscape has many local minima

Let's compute the objective on a 40x40 grid and visualize the landscape.

In [ ]:
l2_vals = np.linspace(0.3, 2.5, 40)
l3_vals = np.linspace(0.3, 2.5, 40)
L2g, L3g = np.meshgrid(l2_vals, l3_vals)
Z = np.vectorize(lambda a, b: objective([a, b]))(L2g, L3g)

fig, ax = plt.subplots(figsize=(7, 5))
cf = ax.contourf(L2g, L3g, np.log1p(Z), levels=30, cmap='viridis')
ax.set_xlabel(r'$\ell_2$ (coupler length)')
ax.set_ylabel(r'$\ell_3$ (follower length)')
ax.set_title('Objective landscape — many basins of attraction')
plt.colorbar(cf, label=r'$\ln(1 + f)$')
plt.tight_layout()

### BFGS from different starting points converges to different minima

We run BFGS from 60 random starting points and collect the final objective values.

In [ ]:
rng = np.random.default_rng(42)
n_starts = 60
starts = rng.uniform(0.3, 2.5, size=(n_starts, 2))

results = []
for x0 in starts:
    res = minimize(objective, x0, method='L-BFGS-B',
                   bounds=[(0.3, 2.5), (0.3, 2.5)])
    results.append(res)

final_vals = np.array([r.fun for r in results])
final_pts = np.array([r.x for r in results])

print(f'Best objective found:  {final_vals.min():.4f}')
print(f'Worst objective found: {final_vals.max():.4f}')
print(f'Median objective:      {np.median(final_vals):.4f}')
print(f'Unique minima (tol=0.05): ~{len(np.unique(np.round(final_vals, 2)))} distinct values')

### Histogram of final objective values

The wide spread confirms that BFGS lands in many different local minima.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(final_vals[final_vals < 100], bins=20, edgecolor='black', alpha=0.7)
ax.axvline(final_vals.min(), color='red', ls='--', label=f'Best = {final_vals.min():.3f}')
ax.set_xlabel('Final objective value')
ax.set_ylabel('Count')
ax.set_title('BFGS from 60 random starts — histogram of results')
ax.legend()
plt.tight_layout()

### Contour plot: start points and where they converge

Arrows show the journey from each starting point to the local minimum BFGS found.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
cf = ax.contourf(L2g, L3g, np.log1p(Z), levels=30, cmap='viridis', alpha=0.8)
plt.colorbar(cf, label=r'$\ln(1 + f)$')

# Draw arrows from start to final point
for x0, res in zip(starts, results):
    dx = res.x[0] - x0[0]
    dy = res.x[1] - x0[1]
    ax.annotate('', xy=res.x, xytext=x0,
                arrowprops=dict(arrowstyle='->', color='white', lw=0.8, alpha=0.6))

# Color final points by objective value
sc = ax.scatter(final_pts[:, 0], final_pts[:, 1], c=final_vals,
                cmap='hot_r', edgecolors='black', s=40, zorder=5,
                vmin=final_vals.min(), vmax=np.percentile(final_vals[final_vals < 100], 90))
ax.scatter(starts[:, 0], starts[:, 1], marker='x', c='cyan', s=20, alpha=0.5, label='starts')

ax.set_xlabel(r'$\ell_2$')
ax.set_ylabel(r'$\ell_3$')
ax.set_title('BFGS from random starts — arrows show convergence to different basins')
ax.legend()
plt.tight_layout()

### Visualizing the best result found

Even with 60 random starts, we may not have found the global optimum.

In [ ]:
best_idx = np.argmin(final_vals)
best_x = final_pts[best_idx]
l2_best, l3_best = best_x

# Compute coupler path at best solution
path_best = np.array([coupler_point(t, l2_best, l3_best) for t in THETAS])

# Visualize mechanism
mechanism = FourBar(L0, L1, l2_best, l3_best)
fig, ax = plt.subplots(figsize=(6, 5))
mechanism.plot(np.deg2rad(90), ax=ax)
ax.plot(TARGETS[:, 0], TARGETS[:, 1], 'r*', ms=10, label='targets')
ax.plot(path_best[:, 0], path_best[:, 1], 'b.-', label='coupler path')
ax.set_aspect('equal')
ax.legend()
ax.set_title(f'Best from 60 BFGS runs: $\\ell_2$={l2_best:.3f}, $\\ell_3$={l3_best:.3f}, f={final_vals[best_idx]:.4f}')
plt.tight_layout()

### Why multi-start is not enough

Multi-start BFGS is a brute-force approach: run many local searches and keep the best. It works sometimes, but:

1. **No information sharing** — each run is independent; we don't learn from previous failures.
2. **Exponential coverage** — in higher dimensions, the number of basins grows rapidly.
3. **No guarantee** — even 1000 starts might miss a narrow global basin.

**Global optimization methods** (GA, PSO, DE) maintain a *population* of solutions that share information and systematically explore the landscape. That is Module 3.

---

## Exercises

1. **More starts:** Increase `n_starts` to 200. Does the best objective improve significantly? How many distinct minima do you find?

2. **Starting region matters:** Restrict starting points to `[0.5, 1.5]` instead of `[0.3, 2.5]`. How does the histogram change?

3. **Nelder-Mead vs BFGS:** Replace `method='L-BFGS-B'` with `method='Nelder-Mead'` (a derivative-free local method). Does it find different minima? Is the spread of final values wider or narrower?

4. **Basin counting:** Round final objective values to 1 decimal place and count unique values. Roughly how many distinct basins does the landscape have in the region $[0.3, 2.5]^2$?